In [ ]:
import os
import json
import random
from pathlib import Path
import numpy as np
import pandas as pd

import torch
import torch.nn as nn
import torch.nn.functional as F

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (
    classification_report, confusion_matrix, roc_auc_score,
    accuracy_score, precision_score, recall_score, f1_score,
    precision_recall_curve, average_precision_score
)
from sklearn.neighbors import kneighbors_graph

from torch_geometric.data import Data
from torch_geometric.nn import GCNConv

# Locate the project root whether Jupyter is launched from the repository root or notebooks/.
_CWD = Path.cwd().resolve()
if (_CWD / "README.md").exists():
    PROJECT_ROOT = _CWD
elif (_CWD.parent / "README.md").exists():
    PROJECT_ROOT = _CWD.parent
else:
    PROJECT_ROOT = _CWD

FIGURE_DIR = PROJECT_ROOT / "figures"
RESULTS_DIR = PROJECT_ROOT / "results"
MODEL_OUTPUT_DIR = PROJECT_ROOT / "outputs" / "models"
for _dir in (FIGURE_DIR, RESULTS_DIR, MODEL_OUTPUT_DIR):
    _dir.mkdir(parents=True, exist_ok=True)

# -----------------------------
# Reproducibility / experiments
# -----------------------------
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

K = 10
GNN_TRAIN_SAMPLE_SIZE = 100000
VAL_SAMPLE_SIZE = 100000
GNN_EPOCHS = 20
GNN_LR = 0.01
RF_ESTIMATORS = 100
RF_MAX_DEPTH = 15
EXPERIMENT_ID = "final_candidate_100k_train_100k_val_nodes"


# AE–GNN–RF IDS: Full Experiment and Feature-Fusion Study

This notebook combines the previously separate base experiment, ablation study, and latent/graph feature-fusion experiment into one sequential workflow.

**Workflow:** CIC-IDS2017 CSVs → cleaning and split → Autoencoder latent vector `Z` (32-D) → k-NN graph (`k=10`) → GCN hidden embedding `H` (16-D) → Random Forest comparisons using `H`, `[Z || H]` (48-D), and `[Z || reconstruction error || H]` (49-D).

**Protocol:** seed `42`; stratified 64% train / 16% validation / 20% test split; preprocessing fitted on training only; 100,000 stratified training graph nodes and 100,000 validation graph nodes; validation-selected F2 thresholds; test graph contains all test flows.

**Research interpretation:** the H-only model is a baseline. The `[Z || H]` model is the principal fused candidate in the recorded results. The AE+RF baseline is slightly stronger overall in the current benchmark, so this repository does not claim that GNN fusion universally outperforms AE+RF.

Run cells from top to bottom in a fresh kernel. The public repository does not include the raw dataset or saved model weights.


In [ ]:
import pandas as pd
from pathlib import Path

files = [
    "Monday-WorkingHours.pcap_ISCX.csv",
    "Tuesday-WorkingHours.pcap_ISCX.csv",
    "Wednesday-workingHours.pcap_ISCX.csv",
    "Thursday-WorkingHours-Morning-WebAttacks.pcap_ISCX.csv",
    "Thursday-WorkingHours-Afternoon-Infilteration.pcap_ISCX.csv",
    "Friday-WorkingHours-Morning.pcap_ISCX.csv",
    "Friday-WorkingHours-Afternoon-PortScan.pcap_ISCX.csv",
    "Friday-WorkingHours-Afternoon-DDos.pcap_ISCX.csv"
]

candidate_dirs = [
    PROJECT_ROOT / "data" / "raw" / "cicids2017",
    Path.cwd(),
    PROJECT_ROOT,
    Path("/content")
]
DATA_DIR = next((d for d in candidate_dirs if all((d / name).exists() for name in files)), None)
if DATA_DIR is None:
    expected = PROJECT_ROOT / "data" / "raw" / "cicids2017"
    raise FileNotFoundError(
        "CIC-IDS2017 source CSVs were not found. Download the eight files listed in README.md "
        f"and place them in: {expected}. Raw data is intentionally not committed to GitHub."
    )

print("Using CIC-IDS2017 CSV directory:", DATA_DIR)
dfs = []
for file in files:
    print("Loading:", file)
    df = pd.read_csv(DATA_DIR / file, encoding="latin1", low_memory=False)
    dfs.append(df)
print("All files loaded successfully")


In [ ]:
import pandas as pd
import numpy as np

# 1. MERGE
data = pd.concat(dfs, ignore_index=True)

print("Merged shape:", data.shape)

# 2. CLEAN COLUMN NAMES (VERY IMPORTANT)
data.columns = data.columns.str.strip()

# 3. REMOVE INF / NaN
data.replace([np.inf, -np.inf], np.nan, inplace=True)
data = data.dropna().reset_index(drop=True)

print("After cleaning:", data.shape)

# 4. CHECK LABEL COLUMN NAME
print("Columns:", data.columns)

# usually it's 'Label'
target_col = "Label"

# 5. LABEL DISTRIBUTION
print("\nRaw label distribution:")
print(data[target_col].value_counts())

In [ ]:
# Convert to binary: BENIGN = 0, ATTACK = 1
data[target_col] = data[target_col].apply(lambda x: 0 if x == "BENIGN" else 1)

print("\nBinary distribution:")
print(data[target_col].value_counts())

In [ ]:
# Define features and target FIRST
X = data.drop(columns=[target_col])
y = data[target_col]

print("X shape:", X.shape)
print("y distribution:\n", y.value_counts())

In [ ]:
# Drop constant columns
X = X.loc[:, X.nunique() > 1]

print("After removing constants:", X.shape)

In [ ]:
# Recombine to remove duplicates safely
df_full = X.copy()
df_full[target_col] = y.values

before = df_full.shape[0]

df_full = df_full.drop_duplicates()

after = df_full.shape[0]

print("Before:", before)
print("After:", after)
print("Removed duplicates:", before - after)

In [ ]:
X = df_full.drop(columns=[target_col])
X = X.select_dtypes(include=[np.number])
y = df_full[target_col]

In [ ]:
from sklearn.model_selection import train_test_split

X = df_full.drop(columns=[target_col])
y = df_full[target_col]

# Outer split: final hold-out test set (never used for model/threshold selection)
X_train_full, X_test, y_train_full, y_test = train_test_split(
    X, y,
    test_size=0.2,
    stratify=y,
    random_state=SEED
)

# Inner split: validation set used only for model-selection / threshold selection
X_train, X_val, y_train, y_val = train_test_split(
    X_train_full, y_train_full,
    test_size=0.2,
    stratify=y_train_full,
    random_state=SEED
)

print("Train:", X_train.shape)
print("Validation:", X_val.shape)
print("Test:", X_test.shape)


In [ ]:
import numpy as np

train_hash = set(map(tuple, X_train.values))
val_hash = set(map(tuple, X_val.values))
test_hash = set(map(tuple, X_test.values))

print("Train/Validation overlap:", len(train_hash & val_hash))
print("Train/Test overlap:", len(train_hash & test_hash))
print("Validation/Test overlap:", len(val_hash & test_hash))


In [ ]:
# Keep only numeric features and fit preprocessing on TRAIN only.
X_train = X_train.select_dtypes(include=[np.number])
X_val = X_val.select_dtypes(include=[np.number])
X_test = X_test.select_dtypes(include=[np.number])

from sklearn.impute import SimpleImputer

imputer = SimpleImputer(strategy="median")

X_train = imputer.fit_transform(X_train)
X_val = imputer.transform(X_val)
X_test = imputer.transform(X_test)


In [ ]:
from sklearn.preprocessing import StandardScaler

# IMPORTANT: scaler is fitted only on training data.
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_val_scaled = scaler.transform(X_val)
X_test_scaled = scaler.transform(X_test)


In [ ]:
#AE features

In [ ]:
import torch
import torch.nn as nn

class AE(nn.Module):
    def __init__(self, input_dim):
        super().__init__()

        # Encoder
        self.encoder = nn.Sequential(
            nn.Linear(input_dim, 128),
            nn.ReLU(),
            nn.Linear(128, 32)
        )

        # Decoder
        self.decoder = nn.Sequential(
            nn.Linear(32, 128),
            nn.ReLU(),
            nn.Linear(128, input_dim)
        )

    def forward(self, x):
        z = self.encoder(x)
        x_recon = self.decoder(z)
        return x_recon, z

In [ ]:
import torch
import torch.nn as nn
from torch.utils.data import TensorDataset, DataLoader
import torch.optim as optim

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

ae_model = AE(input_dim=X_train_scaled.shape[1]).to(device)

criterion = nn.MSELoss()
optimizer = optim.Adam(ae_model.parameters(), lr=1e-3)

X_tensor = torch.tensor(X_train_scaled, dtype=torch.float32).to(device)

losses = []
epochs = 10

for epoch in range(epochs):
    ae_model.train()
    
    optimizer.zero_grad()
    recon, _ = ae_model(X_tensor)
    
    loss = criterion(recon, X_tensor)
    loss.backward()
    optimizer.step()
    losses.append(loss.item())
    
    print(f"Epoch {epoch+1}, Loss: {loss.item():.4f}")

In [ ]:
import matplotlib.pyplot as plt

plt.figure(figsize=(8,5))
plt.plot(losses)
plt.xlabel("Epoch")
plt.ylabel("Reconstruction Loss")
plt.title("Autoencoder Training Loss")
plt.grid(True)

plt.savefig(FIGURE_DIR / "autoencoder_training_loss.png", dpi=300, bbox_inches="tight")
plt.show()

# ============================================================
# GRAPH CONSTRUCTION: k-NN on AUTOENCODER LATENT FEATURES
# ============================================================
# Each traffic flow becomes a node. k-NN in latent space defines edges between
# similar flows. The GNN then learns from this relational structure.


In [ ]:
import torch
import numpy as np

ae_model.eval()

with torch.no_grad():
    X_train_tensor = torch.tensor(X_train_scaled, dtype=torch.float32).to(device)
    X_val_tensor = torch.tensor(X_val_scaled, dtype=torch.float32).to(device)
    X_test_tensor = torch.tensor(X_test_scaled, dtype=torch.float32).to(device)

    train_recon, Z_train = ae_model(X_train_tensor)
    val_recon, Z_val = ae_model(X_val_tensor)
    test_recon, Z_test = ae_model(X_test_tensor)

Z_train = Z_train.cpu().numpy()
Z_val = Z_val.cpu().numpy()
Z_test = Z_test.cpu().numpy()

train_recon = train_recon.cpu().numpy()
val_recon = val_recon.cpu().numpy()
test_recon = test_recon.cpu().numpy()

# Reconstruction error is retained for AE+RF baseline analysis only.
re_train = np.mean(np.square(X_train_scaled - train_recon), axis=1).reshape(-1, 1)
re_val = np.mean(np.square(X_val_scaled - val_recon), axis=1).reshape(-1, 1)
re_test = np.mean(np.square(X_test_scaled - test_recon), axis=1).reshape(-1, 1)

print("Latent shapes:", Z_train.shape, Z_val.shape, Z_test.shape)


In [ ]:
# AE+RF baseline features are kept separately.
ae_rf_train_features = np.hstack([Z_train, re_train])
ae_rf_val_features = np.hstack([Z_val, re_val])
ae_rf_test_features = np.hstack([Z_test, re_test])

print("AE+RF feature shape:", ae_rf_train_features.shape)


In [ ]:
# ============================================================
# STEP 1: SAMPLE GNN TRAINING NODES FROM TRAINING ONLY
# ============================================================
# This v3 controlled experiment uses a 100k-node stratified training graph.
# Validation and test graphs are built separately.

idx_gnn, _ = train_test_split(
    np.arange(len(Z_train)),
    train_size=GNN_TRAIN_SAMPLE_SIZE,
    stratify=y_train,
    random_state=SEED
)

X_gnn_train = Z_train[idx_gnn]
y_gnn_train = y_train.iloc[idx_gnn].to_numpy()

print("GNN training nodes:", X_gnn_train.shape)
print("GNN training labels:", y_gnn_train.shape)


In [ ]:
# ============================================================
# STEP 2: BUILD TRAINING k-NN GRAPH
# ============================================================
def build_knn_graph(X_nodes, y=None, k=K, n_jobs=-1):
    A = kneighbors_graph(
        X_nodes,
        n_neighbors=k,
        mode="connectivity",
        include_self=False,
        n_jobs=n_jobs
    )

    edge_index = torch.tensor(
        np.vstack(A.nonzero()),
        dtype=torch.long
    )

    x = torch.tensor(X_nodes, dtype=torch.float32)

    if y is None:
        return Data(x=x, edge_index=edge_index)

    y_tensor = torch.tensor(np.asarray(y), dtype=torch.long)
    return Data(x=x, edge_index=edge_index, y=y_tensor)

train_graph = build_knn_graph(X_gnn_train, y_gnn_train)
print(train_graph)
print("Edges:", train_graph.edge_index.shape[1])


In [ ]:
# ============================================================
# STEP 3: GNN LEARNS RELATIONAL / CONTEXTUAL PATTERNS
# ============================================================
class GCN(nn.Module):
    def __init__(self, input_dim=32, hidden_dim=16, num_classes=2):
        super().__init__()
        self.conv1 = GCNConv(input_dim, hidden_dim)
        self.conv2 = GCNConv(hidden_dim, num_classes)

    def forward(self, data, return_embedding=False):
        h = self.conv1(data.x, data.edge_index)
        h = F.relu(h)
        logits = self.conv2(h, data.edge_index)

        if return_embedding:
            return logits, h
        return logits


In [ ]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

gcn_model = GCN().to(device)
train_graph = train_graph.to(device)

optimizer = torch.optim.Adam(gcn_model.parameters(), lr=GNN_LR)
losses = []

for epoch in range(GNN_EPOCHS):
    gcn_model.train()
    optimizer.zero_grad()

    out = gcn_model(train_graph)
    loss = F.cross_entropy(out, train_graph.y)
    loss.backward()
    optimizer.step()

    losses.append(loss.item())
    print(f"Epoch {epoch+1:02d} | Loss = {loss.item():.4f}")


In [ ]:
import matplotlib.pyplot as plt

plt.figure(figsize=(7,5))
plt.plot(losses, marker="o")
plt.xlabel("Epoch")
plt.ylabel("Cross-Entropy Loss")
plt.title("GNN Training Loss")
plt.grid(True)
plt.savefig(FIGURE_DIR / "gnn_training_loss_h_only_baseline.png", dpi=300, bbox_inches="tight")
plt.show()


In [ ]:
# Training-graph GNN diagnostic accuracy + hidden node embeddings H.
gcn_model.eval()
with torch.no_grad():
    train_logits, H_train_gnn = gcn_model(train_graph, return_embedding=True)

gnn_train_pred = train_logits.argmax(dim=1)
gnn_train_acc = (gnn_train_pred == train_graph.y).float().mean().item()

H_train_gnn = H_train_gnn.cpu().numpy()

print("GNN training-graph accuracy:", gnn_train_acc)
print("GNN embedding H shape:", H_train_gnn.shape)


In [ ]:
import matplotlib.pyplot as plt

plt.figure(figsize=(5,4))
plt.bar(["GNN"], [gnn_train_acc * 100])
plt.ylabel("Accuracy (%)")
plt.title("GNN Training-Graph Accuracy")
plt.ylim(0, 100)
plt.savefig("gnn_accuracy.png", dpi=300, bbox_inches="tight")
plt.show()


In [ ]:
print("Training graph edge_index shape:", train_graph.edge_index.shape)
print("First 20 edges:\n", train_graph.edge_index[:, :20])


In [ ]:
print("Minimum node index:", train_graph.edge_index.min().item())
print("Maximum node index:", train_graph.edge_index.max().item())


In [ ]:
from sklearn.decomposition import PCA
import matplotlib.pyplot as plt
import numpy as np

sample_size = 500
rng = np.random.default_rng(SEED)
idx = rng.choice(len(Z_train), sample_size, replace=False)

X_vis = Z_train[idx]
y_vis = y_train.iloc[idx].to_numpy()

pca = PCA(n_components=2, random_state=SEED)
X_2d = pca.fit_transform(X_vis)

A = kneighbors_graph(
    X_2d,
    n_neighbors=K,
    mode="connectivity",
    include_self=False
)

plt.figure(figsize=(10,8))
rows, cols = A.nonzero()

for i, j in zip(rows, cols):
    plt.plot([X_2d[i,0], X_2d[j,0]], [X_2d[i,1], X_2d[j,1]],
             color="lightgray", alpha=0.3, linewidth=0.5)

plt.scatter(X_2d[y_vis==0,0], X_2d[y_vis==0,1], c="royalblue", s=25, label="Benign")
plt.scatter(X_2d[y_vis==1,0], X_2d[y_vis==1,1], c="crimson", s=25, label="Attack")
plt.title("k-NN Graph Constructed from Autoencoder Latent Features", fontsize=15, weight="bold")
plt.legend()
plt.tight_layout()
plt.savefig(FIGURE_DIR / "knn_latent_graph_diagnostic.png", dpi=300, bbox_inches="tight")
plt.show()


## Part A — H-only baseline

The first stage reproduces the GNN-hidden-embedding baseline: **network flow features → Autoencoder latent features `Z` → k-NN graph → GCN → 16-D hidden embedding `H` → Random Forest**. The RF sees `H` only in this part. The fusion experiment below then compares `H`, `[Z || H]`, and `[Z || reconstruction error || H]` using the same trained AE/GCN representations.


In [ ]:
# ============================================================
# STEP 4: BUILD A SEPARATE VALIDATION GRAPH
# ============================================================
# Validation is not used to train the GNN or RF. It is used to choose the final
# RF decision threshold. We use a fixed 100k stratified validation graph so the final run matches the v3 controlled protocol.

val_sample_idx, _ = train_test_split(
    np.arange(len(Z_val)),
    train_size=VAL_SAMPLE_SIZE,
    stratify=y_val,
    random_state=SEED
)

X_gnn_val = Z_val[val_sample_idx]
y_gnn_val = y_val.iloc[val_sample_idx].to_numpy()

val_graph = build_knn_graph(X_gnn_val, y_gnn_val)
val_graph_device = val_graph.to(device)

with torch.no_grad():
    val_logits, H_val = gcn_model(val_graph_device, return_embedding=True)

val_gnn_pred = val_logits.argmax(dim=1).cpu().numpy()
val_gnn_acc = accuracy_score(y_gnn_val, val_gnn_pred)
H_val = H_val.cpu().numpy()

print("Validation graph:", val_graph)
print("Validation GNN accuracy:", val_gnn_acc)
print("Validation GNN embedding shape:", H_val.shape)


In [ ]:
# ============================================================
# STEP 5: BUILD THE FINAL TEST GRAPH AND GET GNN EMBEDDINGS
# ============================================================
# IMPORTANT: the test labels are NOT supplied to the GNN. Test labels are kept
# only for the final evaluation below.

test_graph = build_knn_graph(Z_test, y=None)
test_graph_device = test_graph.to(device)

with torch.no_grad():
    test_logits, H_test = gcn_model(test_graph_device, return_embedding=True)

gnn_test_pred = test_logits.argmax(dim=1).cpu().numpy()
H_test = H_test.cpu().numpy()

print("Test graph:", test_graph)
print("Test GNN embedding shape:", H_test.shape)


In [ ]:
# ============================================================
# STEP 6: RANDOM FOREST CLASSIFIER ON GNN OUTPUT H
# ============================================================
# This is the critical hybrid step: RF receives the GNN hidden node embedding H,
# not the original AE features or the GNN's 2-class logits.

rf = RandomForestClassifier(
    n_estimators=RF_ESTIMATORS,
    max_depth=RF_MAX_DEPTH,
    max_features="sqrt",
    n_jobs=-1,
    class_weight="balanced",
    random_state=SEED
)

rf.fit(H_train_gnn, y_gnn_train)

print("RF input shape:", H_train_gnn.shape)
print("RF trained on GNN embeddings H.")


In [ ]:
# HYBRID SANITY CHECK: RF MUST CONSUME GNN H EMBEDDINGS
assert H_train_gnn.shape[1] == 16, f"Unexpected GNN embedding dimension: {H_train_gnn.shape[1]}"
assert rf.n_features_in_ == H_train_gnn.shape[1], "RF is not trained on GNN embeddings H"
print("Verified pipeline: AE -> latent Z -> k-NN graph -> GCN -> H -> RF")
print("RF input dimension:", rf.n_features_in_)


In [ ]:
# Generate validation and test probabilities.
proba_val = rf.predict_proba(H_val)[:, 1]
proba_test = rf.predict_proba(H_test)[:, 1]

print("Validation probabilities:", proba_val.shape)
print("Test probabilities:", proba_test.shape)


In [ ]:
# ============================================================
# STEP 7: SELECT THRESHOLD ON VALIDATION ONLY
# ============================================================
# The original notebook selected the threshold using TEST labels. That was data
# leakage. Here we optimize F2 on validation and freeze the threshold before test.

precision_val, recall_val, thresholds_val = precision_recall_curve(y_gnn_val, proba_val)
f2_val = (1 + 2**2) * (precision_val * recall_val) / (2**2 * precision_val + recall_val + 1e-8)

f2_for_threshold = f2_val[:-1]
best_idx = int(np.nanargmax(f2_for_threshold))
best_thresh = float(thresholds_val[best_idx])

print("Validation-selected threshold (max F2):", best_thresh)
print("Validation precision at selected point:", precision_val[best_idx])
print("Validation recall at selected point:", recall_val[best_idx])


In [ ]:
# Apply the frozen validation-selected threshold to the untouched test set.
pred = (proba_test >= best_thresh).astype(int)

print("Predictions generated on test set using the frozen threshold.")


In [ ]:
# ============================================================
# STEP 8: FINAL TEST EVALUATION
# ============================================================
print(classification_report(y_test, pred, digits=4))
print("Confusion matrix:")
print(confusion_matrix(y_test, pred))
print("ROC-AUC:", roc_auc_score(y_test, proba_test))
print("Accuracy:", accuracy_score(y_test, pred))
print("Precision:", precision_score(y_test, pred))
print("Recall:", recall_score(y_test, pred))
print("F1:", f1_score(y_test, pred))


In [ ]:
print("Frozen validation-selected threshold:", best_thresh)


In [ ]:
#visualization

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
from sklearn.metrics import confusion_matrix

cm = confusion_matrix(y_test, pred)
classes = ["Normal Traffic", "Attack Traffic"]

fig, ax = plt.subplots(figsize=(6, 5))
im = ax.imshow(cm)
cbar = ax.figure.colorbar(im, ax=ax)
cbar.ax.set_ylabel("Number of Samples", rotation=-90, va="bottom")

ax.set_xticks(np.arange(len(classes)))
ax.set_yticks(np.arange(len(classes)))
ax.set_xticklabels(classes)
ax.set_yticklabels(classes)
ax.set_xlabel("Predicted Class", fontsize=12)
ax.set_ylabel("Actual Class", fontsize=12)
ax.set_title("Confusion Matrix of AE–GNN–RF Model", fontsize=13)
plt.setp(ax.get_xticklabels(), rotation=30, ha="right")

threshold = cm.max() / 2.
for i in range(cm.shape[0]):
    for j in range(cm.shape[1]):
        ax.text(j, i, f"{cm[i, j]}", ha="center", va="center",
                color="black" if cm[i, j] > threshold else "white", fontsize=11)

ax.grid(False)
plt.tight_layout()
plt.savefig(FIGURE_DIR / "h_only_confusion_matrix_diagnostic.png", dpi=300, bbox_inches="tight")
plt.savefig(FIGURE_DIR / "h_only_confusion_matrix_diagnostic.eps", format="eps", bbox_inches="tight")
plt.show()


In [ ]:
from sklearn.metrics import roc_curve, auc
import matplotlib.pyplot as plt

fpr, tpr, _ = roc_curve(y_test, proba_test)
roc_auc = auc(fpr, tpr)

plt.figure(figsize=(6,5))
plt.plot(fpr, tpr, linewidth=2, label=f"AUC = {roc_auc:.4f}")
plt.plot([0,1], [0,1], linestyle="--", linewidth=1)
plt.xlabel("False Positive Rate")
plt.ylabel("True Positive Rate")
plt.title("ROC Curve")
plt.legend(loc="lower right")
plt.grid(alpha=0.3)
plt.tight_layout()
plt.savefig(FIGURE_DIR / "h_only_roc_curve_diagnostic.png", dpi=300, bbox_inches="tight")
plt.show()


In [ ]:
from sklearn.metrics import precision_recall_curve, average_precision_score

precision_test, recall_test, _ = precision_recall_curve(y_test, proba_test)
ap_score = average_precision_score(y_test, proba_test)

plt.figure(figsize=(6,5))
plt.plot(recall_test, precision_test, linewidth=2, label=f"AP = {ap_score:.4f}")
plt.xlabel("Recall")
plt.ylabel("Precision")
plt.title("Precision-Recall Curve")
plt.legend(loc="lower left")
plt.grid(alpha=0.3)
plt.tight_layout()
plt.savefig(FIGURE_DIR / "h_only_pr_curve_diagnostic.png", dpi=300, bbox_inches="tight")
plt.show()


In [ ]:
import pandas as pd
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, roc_auc_score

results = pd.DataFrame({
    "Metric": [
        "Accuracy",
        "Precision (Attack)",
        "Recall (Attack)",
        "F1-score (Attack)",
        "ROC-AUC"
    ],
    "Value": [
        accuracy_score(y_test, pred),
        precision_score(y_test, pred),
        recall_score(y_test, pred),
        f1_score(y_test, pred),
        roc_auc_score(y_test, proba_test)
    ]
})

results["Value"] = results["Value"].apply(lambda x: f"{x:.4f}")
results


In [ ]:
import pandas as pd
import matplotlib.pyplot as plt

# Count binary classes
class_counts = pd.Series(y).value_counts().sort_index()

plt.figure(figsize=(10,6))

bars = plt.bar(
    ['Benign Traffic', 'Attack Traffic'],
    class_counts.values,
    color=['#3498DB', '#E74C3C'],
    edgecolor='black',
    linewidth=1.5
)

# Add values on bars
for bar in bars:
    plt.text(
        bar.get_x() + bar.get_width()/2,
        bar.get_height() + 500,
        f'{int(bar.get_height()):,}',
        ha='center',
        fontsize=11,
        fontweight='bold'
    )

plt.title('Dataset Class Distribution (CIC-IDS2017)',
          fontsize=14,
          fontweight='bold')

plt.xlabel('Traffic Class')
plt.ylabel('Number of Samples')

plt.grid(axis='y', linestyle='--', alpha=0.4)

plt.tight_layout()

plt.savefig(FIGURE_DIR / 'dataset_class_distribution_diagnostic.png', dpi=300, bbox_inches='tight')

plt.show()

In [ ]:
# Dataset class distribution using the ACTUAL processed dataset.
import matplotlib.pyplot as plt

class_counts = pd.Series(y).value_counts().sort_index()
labels = ["Benign Traffic", "Attack Traffic"]

plt.figure(figsize=(7,7))
plt.pie(
    class_counts.values,
    labels=labels,
    autopct="%1.1f%%",
    startangle=90,
    explode=(0.05, 0.05),
    shadow=True
)
plt.title("Class Distribution of Processed CIC-IDS2017 Dataset", fontsize=14, fontweight="bold")
plt.tight_layout()
plt.savefig("Figure_5_1_PieChart.png", dpi=600, bbox_inches="tight")
plt.show()


In [ ]:
import os
import json
import torch
import joblib

MODEL_OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

# Save trained components.
torch.save(ae_model.state_dict(), MODEL_OUTPUT_DIR / "autoencoder.pth")
torch.save(gcn_model.state_dict(), MODEL_OUTPUT_DIR / "gcn_model.pth")
joblib.dump(rf, MODEL_OUTPUT_DIR / "random_forest_h_only.pkl")

# Save the final evaluation threshold and key experiment settings so the pipeline
# can be reproduced without re-selecting the threshold on test data.
experiment_config = {
    "experiment_id": EXPERIMENT_ID,
    "protocol": "seed=42; stratified 64/16/20 train/validation/test; 100k stratified GNN training graph; 100k validation graph; full test graph; threshold selected on validation only",
    "seed": SEED,
    "latent_dim": int(Z_train.shape[1]),
    "knn_k": K,
    "gnn_training_nodes": int(len(H_train_gnn)),
    "validation_graph_nodes": int(len(H_val)),
    "gnn_hidden_dim": int(H_train_gnn.shape[1]),
    "gnn_epochs": GNN_EPOCHS,
    "gnn_learning_rate": GNN_LR,
    "rf_estimators": RF_ESTIMATORS,
    "rf_max_depth": RF_MAX_DEPTH,
    "decision_threshold": best_thresh,
    "threshold_selection": "maximum F2 on validation graph",
    "test_size": int(len(y_test))
}

with open(MODEL_OUTPUT_DIR / "experiment_config_h_only.json", "w") as f:
    json.dump(experiment_config, f, indent=2)

print("Saved AE, GCN and H-only RF checkpoints locally under outputs/models (ignored by Git).")


## Baseline freeze checkpoint

This configuration freezes the H-only baseline protocol: **100,000 stratified GNN training nodes**, **100,000 validation graph nodes**, seed `42`, `k=10`, GCN 32→16→2, and RF with 100 trees / max depth 15. The later fusion sections are separate comparison variants. Since multiple variants are compared on the recorded test partition, treat these as benchmark results needing future external/cross-day validation rather than proof of universal generalization.


## Controlled-protocol note

The configured validation graph has **100,000 nodes** (not 50,000). Thresholds are selected with validation data only; the same held-out test partition is used to report the benchmark comparisons across model variants.


In [ ]:
# ============================================================
# FINAL CANDIDATE FREEZE CHECK
# ============================================================
print("Experiment ID:", EXPERIMENT_ID)
print("Seed:", SEED)
print("GNN training nodes:", len(H_train_gnn))
print("Validation graph nodes:", len(H_val))
print("Test graph nodes:", len(H_test))
print("RF input dimension:", rf.n_features_in_)
print("Decision threshold:", best_thresh)

assert SEED == 42
assert len(H_train_gnn) == 100000
assert len(H_val) == 100000
assert len(H_test) == len(y_test)
assert rf.n_features_in_ == 16

print("✅ FINAL CANDIDATE CHECK PASSED")


## Part B — Ablation study
This section compares AE reconstruction error, AE+RF, AE+GNN, and RF(H). All thresholds are selected from validation scores; test metrics are reported for comparison.


In [ ]:
# ============================================================
# 0. Sanity checks — do NOT skip
# ============================================================
required = [
    "SEED", "K", "GNN_TRAIN_SAMPLE_SIZE", "VAL_SAMPLE_SIZE",
    "Z_train", "Z_val", "Z_test",
    "re_train", "re_val", "re_test",
    "idx_gnn", "y_gnn_train", "y_gnn_val",
    "train_graph", "val_graph", "test_graph",
    "gcn_model", "H_train_gnn", "H_val", "H_test"
]

missing = [name for name in required if name not in globals()]
if missing:
    raise RuntimeError(
        "Missing v3 variables: " + ", ".join(missing) +
        ". These cells must be run after the completed v3 experiment."
    )

actual_gnn_nodes = len(y_gnn_train)

print("Configured GNN training nodes:", GNN_TRAIN_SAMPLE_SIZE)
print("Actual GNN nodes:", actual_gnn_nodes)
print("GNN embedding dimension:", H_train_gnn.shape[1])

assert actual_gnn_nodes == GNN_TRAIN_SAMPLE_SIZE, (
    f"STALE/INCONSISTENT FINAL RUN: configured {GNN_TRAIN_SAMPLE_SIZE} nodes "
    f"but executed graph has {actual_gnn_nodes}. "
    "Rerun v3 from the beginning before running the ablation."
)

actual_val_nodes = len(y_gnn_val)

print("Actual validation graph nodes:", actual_val_nodes)

assert actual_gnn_nodes == 100000
assert actual_val_nodes == VAL_SAMPLE_SIZE
assert H_train_gnn.shape[0] == actual_gnn_nodes
assert H_train_gnn.shape[1] == 16
assert H_val.shape[1] == 16
assert H_test.shape[1] == 16

print("✅ Final-candidate experiment state is internally consistent.")


In [ ]:
# ============================================================
# 1. Common metric/threshold helpers
# ============================================================
import numpy as np
import pandas as pd

from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    roc_auc_score, average_precision_score, confusion_matrix,
    precision_recall_curve, balanced_accuracy_score
)

def select_f2_threshold(y_val, score_val):
    precision, recall, thresholds = precision_recall_curve(y_val, score_val)

    if len(thresholds) == 0:
        return 0.5, np.nan, np.nan, np.nan

    f2 = (
        5.0 * precision[:-1] * recall[:-1]
        / (4.0 * precision[:-1] + recall[:-1] + 1e-12)
    )

    best_idx = int(np.nanargmax(f2))
    threshold = float(thresholds[best_idx])

    return (
        threshold,
        float(precision[best_idx]),
        float(recall[best_idx]),
        float(f2[best_idx]),
    )

def evaluate_binary_model(name, y_val, score_val, y_test, score_test):
    threshold, val_p, val_r, val_f2 = select_f2_threshold(y_val, score_val)
    pred = (score_test >= threshold).astype(int)

    tn, fp, fn, tp = confusion_matrix(y_test, pred).ravel()

    result = {
        "Model": name,
        "Threshold": threshold,
        "Validation_Precision": val_p,
        "Validation_Recall": val_r,
        "Validation_F2": val_f2,
        "Accuracy": accuracy_score(y_test, pred),
        "Precision": precision_score(y_test, pred, zero_division=0),
        "Recall": recall_score(y_test, pred, zero_division=0),
        "F1": f1_score(y_test, pred, zero_division=0),
        "ROC_AUC": roc_auc_score(y_test, score_test),
        "PR_AUC": average_precision_score(y_test, score_test),
        "FPR": fp / (fp + tn),
        "FNR": fn / (fn + tp),
        "Balanced_Accuracy": balanced_accuracy_score(y_test, pred),
        "TN": tn,
        "FP": fp,
        "FN": fn,
        "TP": tp,
    }

    return result, pred


In [ ]:
# ============================================================
# 2. Ablation A — AE only
# ============================================================
# Higher reconstruction error = more anomalous.
ae_score_val = np.asarray(re_val).ravel()
ae_score_test = np.asarray(re_test).ravel()

ae_result, ae_pred = evaluate_binary_model(
    "AE",
    y_val.to_numpy(),
    ae_score_val,
    y_test.to_numpy(),
    ae_score_test
)

print("AE threshold:", ae_result["Threshold"])
print("AE metrics:", {k: ae_result[k] for k in [
    "Accuracy", "Precision", "Recall", "F1", "ROC_AUC", "PR_AUC", "FPR", "FNR"
]})


In [ ]:
# ============================================================
# 3. Ablation B — AE + RF
# ============================================================
# Use the SAME GNN training-node indices for the RF training subset.
# This keeps RF training population identical to the proposed model's RF stage.
ae_rf_train = np.hstack([Z_train[idx_gnn], re_train[idx_gnn]])
y_ae_rf_train = y_gnn_train

ae_rf_val = np.hstack([Z_val, re_val])
ae_rf_test = np.hstack([Z_test, re_test])

ae_rf = RandomForestClassifier(
    n_estimators=100,
    max_depth=15,
    max_features="sqrt",
    n_jobs=-1,
    class_weight="balanced",
    random_state=SEED
)

ae_rf.fit(ae_rf_train, y_ae_rf_train)

ae_rf_score_val = ae_rf.predict_proba(ae_rf_val)[:, 1]
ae_rf_score_test = ae_rf.predict_proba(ae_rf_test)[:, 1]

ae_rf_result, ae_rf_pred = evaluate_binary_model(
    "AE+RF",
    y_val.to_numpy(),
    ae_rf_score_val,
    y_test.to_numpy(),
    ae_rf_score_test
)

print("AE+RF threshold:", ae_rf_result["Threshold"])
print("AE+RF metrics:", {k: ae_rf_result[k] for k in [
    "Accuracy", "Precision", "Recall", "F1", "ROC_AUC", "PR_AUC", "FPR", "FNR"
]})


In [ ]:
# ============================================================
# 4. Ablation C — AE + GNN
# ============================================================
# Use the already-trained GCN as a direct classifier.
# This measures the contribution of the GNN without the RF stage.

device = next(gcn_model.parameters()).device

with torch.no_grad():
    val_logits = gcn_model(val_graph.to(device))
    test_logits = gcn_model(test_graph.to(device))

gnn_score_val = torch.softmax(val_logits, dim=1)[:, 1].cpu().numpy()
gnn_score_test = torch.softmax(test_logits, dim=1)[:, 1].cpu().numpy()

ae_gnn_result, ae_gnn_pred = evaluate_binary_model(
    "AE+GNN",
    np.asarray(y_gnn_val),
    gnn_score_val,
    y_test.to_numpy(),
    gnn_score_test
)

print("AE+GNN threshold:", ae_gnn_result["Threshold"])
print("AE+GNN metrics:", {k: ae_gnn_result[k] for k in [
    "Accuracy", "Precision", "Recall", "F1", "ROC_AUC", "PR_AUC", "FPR", "FNR"
]})


In [ ]:
# ============================================================
# 5. Ablation D — Proposed AE + GNN + RF
# ============================================================
# RF receives H, the 16-D hidden GNN embedding.
hybrid_score_val = rf.predict_proba(H_val)[:, 1]
hybrid_score_test = rf.predict_proba(H_test)[:, 1]

hybrid_result, hybrid_pred = evaluate_binary_model(
    "AE+GNN+RF",
    np.asarray(y_gnn_val),
    hybrid_score_val,
    y_test.to_numpy(),
    hybrid_score_test
)

print("AE+GNN+RF threshold:", hybrid_result["Threshold"])
print("AE+GNN+RF metrics:", {k: hybrid_result[k] for k in [
    "Accuracy", "Precision", "Recall", "F1", "ROC_AUC", "PR_AUC", "FPR", "FNR"
]})


In [ ]:
# ============================================================
# 6. Final ablation table
# ============================================================
results = pd.DataFrame([
    ae_result,
    ae_rf_result,
    ae_gnn_result,
    hybrid_result
])

display_cols = [
    "Model",
    "Accuracy",
    "Precision",
    "Recall",
    "F1",
    "ROC_AUC",
    "PR_AUC",
    "FPR",
    "FNR",
    "Balanced_Accuracy",
    "Threshold",
    "TN", "FP", "FN", "TP"
]

ablation_table = results[display_cols].copy()

pct_cols = [
    "Accuracy", "Precision", "Recall", "F1",
    "ROC_AUC", "PR_AUC", "FPR", "FNR", "Balanced_Accuracy"
]

ablation_table[pct_cols] = ablation_table[pct_cols] * 100

print("Final ablation results (%) except threshold and confusion-matrix counts:")
display(ablation_table)

ablation_table.to_csv(RESULTS_DIR / "ae_gnn_rf_ablation_results_v1.csv", index=False)

print("✅ Saved: ae_gnn_rf_ablation_results_v1.csv")


In [ ]:
# ============================================================
# 7. Interpretation helpers
# ============================================================
full = results.set_index("Model")
ae_rf = full.loc["AE+RF"]
ae_gnn = full.loc["AE+GNN"]
hybrid = full.loc["AE+GNN+RF"]

print("Change from AE+RF -> full hybrid:")
for metric in ["Accuracy", "Precision", "Recall", "F1", "ROC_AUC", "PR_AUC", "FPR", "FNR"]:
    delta = hybrid[metric] - ae_rf[metric]
    print(f"{metric:20s}: {delta:+.6f}")

print("\nChange from AE+GNN -> full hybrid:")
for metric in ["Accuracy", "Precision", "Recall", "F1", "ROC_AUC", "PR_AUC", "FPR", "FNR"]:
    delta = hybrid[metric] - ae_gnn[metric]
    print(f"{metric:20s}: {delta:+.6f}")


## Part C — Feature-fusion experiment: `[Z || H]` and `[Z || reconstruction error || H]`
Here, `Z` is the 32-D AE latent vector and `H` is the 16-D GCN hidden embedding. RF input sizes are 48 and 49 respectively. This is the key experiment behind the reported `[Z || H]` result. The AE and GCN are not retrained in this section.


In [ ]:
# ============================================================
# 0. Sanity checks — final candidate state required
# ============================================================
required = [
    'SEED', 'K', 'GNN_TRAIN_SAMPLE_SIZE', 'VAL_SAMPLE_SIZE',
    'Z_train', 'Z_val', 'Z_test',
    're_train', 're_val', 're_test',
    'idx_gnn', 'y_gnn_train', 'y_gnn_val', 'val_sample_idx',
    'H_train_gnn', 'H_val', 'H_test',
    'y_test'
]

missing = [name for name in required if name not in globals()]
if missing:
    raise RuntimeError(
        'Missing final-candidate variables: ' + ', '.join(missing) +
        '. Run the completed final-candidate notebook first.'
    )

print('Seed:', SEED)
print('GNN training nodes:', len(y_gnn_train))
print('Validation graph nodes:', len(y_gnn_val))
print('Test nodes:', len(y_test))
print('H_train shape:', H_train_gnn.shape)
print('H_val shape:', H_val.shape)
print('H_test shape:', H_test.shape)

assert SEED == 42
assert len(y_gnn_train) == 100000
assert len(y_gnn_val) == 100000
assert H_train_gnn.shape == (100000, 16)
assert H_val.shape == (100000, 16)
assert H_test.shape[0] == len(y_test)
assert H_test.shape[1] == 16

print('✅ Final-candidate state verified.')


In [ ]:
# ============================================================
# 1. Build exactly aligned feature-fusion matrices
# ============================================================
Z_train_gnn = Z_train[idx_gnn]
re_train_gnn = np.asarray(re_train)[idx_gnn].reshape(-1, 1)

Z_val_gnn = Z_val[val_sample_idx]
re_val_gnn = np.asarray(re_val)[val_sample_idx].reshape(-1, 1)

Z_test_full = Z_test
re_test_full = np.asarray(re_test).reshape(-1, 1)

fusion1_train = np.hstack([Z_train_gnn, H_train_gnn])
fusion1_val = np.hstack([Z_val_gnn, H_val])
fusion1_test = np.hstack([Z_test_full, H_test])

fusion2_train = np.hstack([Z_train_gnn, re_train_gnn, H_train_gnn])
fusion2_val = np.hstack([Z_val_gnn, re_val_gnn, H_val])
fusion2_test = np.hstack([Z_test_full, re_test_full, H_test])

print('Fusion-1 [Z,H] shapes:', fusion1_train.shape, fusion1_val.shape, fusion1_test.shape)
print('Fusion-2 [Z,re,H] shapes:', fusion2_train.shape, fusion2_val.shape, fusion2_test.shape)

assert fusion1_train.shape[1] == 48
assert fusion2_train.shape[1] == 49
print('✅ Feature alignment verified.')


In [ ]:
# ============================================================
# 2. Common RF + validation-threshold helper
# ============================================================
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    roc_auc_score, average_precision_score, confusion_matrix,
    precision_recall_curve, balanced_accuracy_score
)

def select_f2_threshold(y_val_local, score_val_local):
    precision, recall, thresholds = precision_recall_curve(
        y_val_local, score_val_local
    )
    if len(thresholds) == 0:
        return 0.5
    f2 = 5.0 * precision[:-1] * recall[:-1] / (
        4.0 * precision[:-1] + recall[:-1] + 1e-12
    )
    return float(thresholds[int(np.nanargmax(f2))])

def run_fusion_experiment(name, Xtr, ytr, Xv, yv, Xte, yte):
    model = RandomForestClassifier(
        n_estimators=100,
        max_depth=15,
        max_features='sqrt',
        n_jobs=-1,
        class_weight='balanced',
        random_state=SEED
    )
    model.fit(Xtr, ytr)

    score_val = model.predict_proba(Xv)[:, 1]
    score_test = model.predict_proba(Xte)[:, 1]
    threshold = select_f2_threshold(yv, score_val)
    pred_test = (score_test >= threshold).astype(int)

    tn, fp, fn, tp = confusion_matrix(yte, pred_test).ravel()
        
    return model, {
        'Model': name,
        'Threshold': threshold,
        'Accuracy': accuracy_score(yte, pred_test),
        'Precision': precision_score(yte, pred_test, zero_division=0),
        'Recall': recall_score(yte, pred_test, zero_division=0),
        'F1': f1_score(yte, pred_test, zero_division=0),
        'ROC_AUC': roc_auc_score(yte, score_test),
        'PR_AUC': average_precision_score(yte, score_test),
        'FPR': fp / (fp + tn),
        'FNR': fn / (fn + tp),
        'Balanced_Accuracy': balanced_accuracy_score(yte, pred_test),
        'TN': tn, 'FP': fp, 'FN': fn, 'TP': tp
    }, pred_test


In [ ]:
# ============================================================
# 3. Fusion-1: RF([Z,H])
# ============================================================
rf_z_h, result_z_h, pred_z_h = run_fusion_experiment(
    'AE+GNN+RF [Z,H]',
    fusion1_train, y_gnn_train,
    fusion1_val, np.asarray(y_gnn_val),
    fusion1_test, np.asarray(y_test)
)

print('Fusion-1 complete:', result_z_h)


In [ ]:
# ============================================================
# 4. Fusion-2: RF([Z,reconstruction_error,H])
# ============================================================
rf_z_re_h, result_z_re_h, pred_z_re_h = run_fusion_experiment(
    'AE+GNN+RF [Z,re,H]',
    fusion2_train, y_gnn_train,
    fusion2_val, np.asarray(y_gnn_val),
    fusion2_test, np.asarray(y_test)
)

print('Fusion-2 complete:', result_z_re_h)


In [ ]:
# ============================================================
# 5. Compare against the frozen baselines/results
# ============================================================
comparison = pd.DataFrame([
    {
        'Model': 'AE+RF',
        'Accuracy': 0.97930989,
        'Precision': 0.91275090,
        'Recall': 0.98946196,
        'F1': 0.94955965,
        'ROC_AUC': 0.99842192,
        'PR_AUC': 0.99363029,
        'FPR': 0.02317794,
        'FNR': 0.01053804
    },
    {
        'Model': 'AE+GNN+RF [H]',
        'Accuracy': 0.96122263,
        'Precision': 0.84823313,
        'Recall': 0.97796265,
        'F1': 0.90849002,
        'ROC_AUC': 0.99523147,
        'PR_AUC': 0.98266651,
        'FPR': 0.04287963,
        'FNR': 0.02203735
    },
    {
        'Model': result_z_h['Model'],
        'Accuracy': result_z_h['Accuracy'],
        'Precision': result_z_h['Precision'],
        'Recall': result_z_h['Recall'],
        'F1': result_z_h['F1'],
        'ROC_AUC': result_z_h['ROC_AUC'],
        'PR_AUC': result_z_h['PR_AUC'],
        'FPR': result_z_h['FPR'],
        'FNR': result_z_h['FNR']
    },
    {
        'Model': result_z_re_h['Model'],
        'Accuracy': result_z_re_h['Accuracy'],
        'Precision': result_z_re_h['Precision'],
        'Recall': result_z_re_h['Recall'],
        'F1': result_z_re_h['F1'],
        'ROC_AUC': result_z_re_h['ROC_AUC'],
        'PR_AUC': result_z_re_h['PR_AUC'],
        'FPR': result_z_re_h['FPR'],
        'FNR': result_z_re_h['FNR']
    }
])

print('Comparison (all values are fractions):')
display(comparison)

comparison.to_csv(RESULTS_DIR / 'ae_gnn_rf_feature_fusion_results_v1.csv', index=False)
print('✅ Saved: ae_gnn_rf_feature_fusion_results_v1.csv')


In [ ]:
# ============================================================
# 6. Direct decision check
# ============================================================
for _, row in comparison.iterrows():
    print(
        f"{row['Model']}: Accuracy={row['Accuracy']:.4f}, "
        f"Precision={row['Precision']:.4f}, "
        f"Recall={row['Recall']:.4f}, "
        f"F1={row['F1']:.4f}, "
        f"ROC-AUC={row['ROC_AUC']:.4f}"
    )
        
best = comparison.loc[comparison['F1'].idxmax()]
print('\nHighest F1 in this controlled comparison:', best['Model'])
        
if best['Model'] in ['AE+GNN+RF [Z,H]', 'AE+GNN+RF [Z,re,H]']:
    print('✅ Feature fusion is a promising final hybrid candidate.')
else:
    print('ℹ️ Feature fusion did not beat the strongest current baseline; keep the result as evidence for the discussion.')


In [ ]:
# ============================================================
# 7. Save final [Z || H] metrics and figures (the principal fused candidate)
# ============================================================
from sklearn.metrics import ConfusionMatrixDisplay, roc_curve, auc, precision_recall_curve, average_precision_score

final_threshold_zh = float(result_z_h["Threshold"])
score_test_zh = rf_z_h.predict_proba(fusion1_test)[:, 1]
cm_zh = confusion_matrix(np.asarray(y_test), pred_z_h)

final_zh_table = pd.DataFrame([result_z_h])
final_zh_table.to_csv(RESULTS_DIR / "final_model_metrics.csv", index=False)

plt.figure(figsize=(6, 5))
plt.imshow(cm_zh)
plt.title("AE–GNN–RF Feature Fusion [Z || H] — Confusion Matrix")
plt.xlabel("Predicted class")
plt.ylabel("Actual class")
plt.xticks([0, 1], ["Benign", "Attack"])
plt.yticks([0, 1], ["Benign", "Attack"])
for i in range(2):
    for j in range(2):
        plt.text(j, i, f"{cm_zh[i, j]:,}", ha="center", va="center")
plt.tight_layout()
plt.savefig(FIGURE_DIR / "final_zh_confusion_matrix_from_notebook.png", dpi=300, bbox_inches="tight")
plt.show()

fpr_zh, tpr_zh, _ = roc_curve(np.asarray(y_test), score_test_zh)
roc_auc_zh = auc(fpr_zh, tpr_zh)
plt.figure(figsize=(6, 5))
plt.plot(fpr_zh, tpr_zh, label=f"ROC-AUC = {roc_auc_zh:.5f}")
plt.plot([0, 1], [0, 1], linestyle="--")
plt.xlabel("False-positive rate")
plt.ylabel("True-positive rate / attack recall")
plt.title("ROC Curve — AE–GNN–RF [Z || H]")
plt.legend(loc="lower right")
plt.grid(alpha=0.3)
plt.tight_layout()
plt.savefig(FIGURE_DIR / "final_zh_roc_curve_from_notebook.png", dpi=300, bbox_inches="tight")
plt.show()

precision_zh, recall_zh, _ = precision_recall_curve(np.asarray(y_test), score_test_zh)
ap_zh = average_precision_score(np.asarray(y_test), score_test_zh)
plt.figure(figsize=(6, 5))
plt.plot(recall_zh, precision_zh, label=f"Average precision = {ap_zh:.5f}")
plt.xlabel("Attack recall")
plt.ylabel("Attack precision")
plt.title("Precision–Recall Curve — AE–GNN–RF [Z || H]")
plt.legend(loc="lower left")
plt.grid(alpha=0.3)
plt.tight_layout()
plt.savefig(FIGURE_DIR / "final_zh_pr_curve_from_notebook.png", dpi=300, bbox_inches="tight")
plt.show()

print("Principal fused-model threshold:", final_threshold_zh)
print("Final [Z || H] metrics written to:", RESULTS_DIR / "final_model_metrics.csv")

# Optional local checkpoint for the [Z || H] RF. Model files are intentionally ignored by Git.
import joblib
joblib.dump(rf_z_h, MODEL_OUTPUT_DIR / "random_forest_zh_fusion.pkl")
with open(MODEL_OUTPUT_DIR / "experiment_config_zh_fusion.json", "w") as _f:
    json.dump({
        "experiment_id": "ae_gnn_rf_zh_fusion_seed42",
        "seed": SEED,
        "input_features": int(X_train_scaled.shape[1]),
        "ae_latent_dim": int(Z_train.shape[1]),
        "gcn_hidden_dim": int(H_train_gnn.shape[1]),
        "rf_fusion_dimension": int(fusion1_train.shape[1]),
        "knn_k": K,
        "gnn_training_nodes": int(len(y_gnn_train)),
        "validation_graph_nodes": int(len(y_gnn_val)),
        "test_nodes": int(len(y_test)),
        "decision_threshold": final_threshold_zh,
        "threshold_selection": "maximum F2 on validation graph",
        "note": "Exploratory benchmark result; full test partition was used for comparison among variants. Validate on an independent source/day before broader generalization claims."
    }, _f, indent=2)
print("Saved [Z || H] RF and config locally under outputs/models (ignored by Git).")


In [ ]:
# ============================================================
# 8. Rebuild the unified comparison table from this execution
# ============================================================
base_comparison = results.copy()
base_comparison["Model"] = base_comparison["Model"].replace({"AE+GNN+RF": "AE+GNN+RF [H]"})
zh_row = {key: result_z_h.get(key, np.nan) for key in base_comparison.columns}
zh_row["Model"] = "AE+GNN+RF [Z,H]"
comparison_all = pd.concat([base_comparison, pd.DataFrame([zh_row])], ignore_index=True)
comparison_all = comparison_all[[
    "Model", "Accuracy", "Precision", "Recall", "F1", "ROC_AUC", "PR_AUC",
    "FPR", "FNR", "Balanced_Accuracy", "Threshold", "TN", "FP", "FN", "TP"
]]
comparison_all.to_csv(RESULTS_DIR / "model_comparison.csv", index=False)
print("Unified comparison saved to:", RESULTS_DIR / "model_comparison.csv")
print("Scores are fractions from 0 to 1; confusion-matrix entries are counts.")
display(comparison_all)
